<a href="https://colab.research.google.com/github/kuds/rl-doom/blob/main/notebooks/02_dqn_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — DQN Training (Stable-Baselines3)

Train a **DQN** agent (via `stable_baselines3.DQN`, `CnnPolicy`) on four
ViZDoom scenarios: *Basic*, *Deadly Corridor*, *Defend the Center*, and
*Deathmatch*.

**Config-driven.** All hyperparameters, env settings, and training budgets
come from `configs/dqn_<scenario>.yaml` so the notebook and the standalone
YAML files can never drift apart. Tune a scenario by editing its YAML; the
notebook just iterates over the configs and calls `train_sb3`.

**Design notes.** Each scenario trains end-to-end and writes its full
artifact bundle (learning curves, eval curve, video, checkpoint, stage
summary) to disk **before** the next scenario starts, so a Colab timeout
or an error on scenario N+1 still leaves scenario N fully shippable.

**Colab baseline.** Defaults target an **L4 GPU + high-memory runtime**:
single `DummyVecEnv` worker (SB3's DQN only supports `n_envs=1`),
`buffer_size=100_000` frames, `batch_size=64`, `learning_starts=10_000`.

## 1. Setup

In [ ]:
# --- Environment setup ---
# Works both locally and on Google Colab: `setup_colab` clones + installs the
# repo when a Colab runtime is detected and is a no-op otherwise, so there is
# nothing to uncomment or edit. Locally, run `pip install -e ".[notebooks]"`
# once beforehand.
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

from rl_doom.utils import setup_colab
setup_colab(extras="notebooks")

import numpy as np
import torch

from rl_doom.paths import load_yaml_config, new_run_dir, write_config
from rl_doom.sb3_utils import gpu_info, policy_kwargs_from_config, train_sb3

torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Global RNG seeding for notebook-level reproducibility. SB3 seeds
# itself per-run via the ``seed=`` kwarg we forward into ``train_sb3``;
# this block covers non-SB3 randomness (numpy, Python ``random``,
# torch) that the notebook may consume outside the training loop.
import random as _random
GLOBAL_SEED = 42
_random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)
GPU_INFO = gpu_info()
print(f"Using device: {DEVICE}")
for k, v in GPU_INFO.items():
    print(f"  {k}: {v}")

In [ ]:
# --- Google Drive persistence (Colab only) ---
# No-op off Colab. On Colab this mounts Drive and symlinks the
# `training_jobs/` and `analysis/` trees into it so runs survive the runtime
# being recycled. Point DRIVE_ROOT wherever you want the artifacts to live.
from rl_doom.utils import setup_google_drive

DRIVE_ROOT = "/content/drive/MyDrive/rl-doom"
setup_google_drive(DRIVE_ROOT)


## 2. Load per-scenario configs

Each scenario's hyperparameters, env settings, and training budget live
in `configs/dqn_<scenario>.yaml`. We load all four upfront so the loop
below can iterate without touching the YAML files again.

In [ ]:
# Flip to True to run the paired curriculum variants for the scenarios
# that have them (deadly_corridor = skill curriculum, deathmatch =
# bot-count curriculum). Every other scenario stays on its baseline YAML.
USE_CURRICULUM = False

SCENARIOS = ["basic", "deadly_corridor", "defend_the_center", "deathmatch"]

# ``load_yaml_config`` accepts either a full path or the stem (e.g.
# ``"dqn_basic"``), resolving the latter against the repo's ``configs/`` dir.
CONFIGS = {s: load_yaml_config(f"dqn_{s}") for s in SCENARIOS}

if USE_CURRICULUM:
    for s in ("deadly_corridor", "deathmatch"):
        CONFIGS[s] = load_yaml_config(f"dqn_{s}_curriculum")
        print(f"[curriculum] {s} -> dqn_{s}_curriculum")

# Quick sanity dump so the Colab cell output shows the knobs that will
# actually drive training.
for s, cfg in CONFIGS.items():
    hp = cfg["hyperparams"]
    training = cfg["training"]
    variant = "curriculum" if cfg.get("curriculum") else "baseline"
    print(
        f"{s:20s}  [{variant:10s}]  total_timesteps={training['total_timesteps']:>10,}  "
        f"lr={hp['lr']:.1e}  buffer={hp['buffer_size']:>7,}  "
        f"learning_starts={hp['learning_starts']:>6,}"
    )

## 3. Train each scenario (artifacts written per scenario)

`train_sb3` does the full end-to-end training for one scenario. SB3's DQN
requires `n_envs=1`, so the replay buffer is populated in a single worker.
For each scenario we write the checkpoint, learning curves, eval curve,
gameplay video, and stage summary **before** moving on to the next one.

The loop pulls everything it needs from the YAML: hyperparams,
`env_settings` (resize/frame_skip/num_stack/doom_skill/num_bots),
`policy_kwargs` (features_dim / net_arch), seed, and training/eval
schedules. `config.json` in each run directory records the full merged
view so the run is reproducible from a single file.

In [ ]:
from IPython.display import Video, display

results = {}
for scenario in SCENARIOS:
    cfg = CONFIGS[scenario]
    hp = dict(cfg["hyperparams"])
    env_cfg = dict(cfg.get("env", {}))
    policy_cfg = dict(cfg.get("policy", {}))
    training_cfg = cfg["training"]
    eval_cfg = cfg.get("eval", {})
    curriculum_cfg = cfg.get("curriculum")
    seed = int(cfg.get("seed", 42))
    total_ts = int(training_cfg["total_timesteps"])
    n_envs = int(env_cfg.get("n_envs", 1))

    # Tag curriculum runs so they don't collide with baseline run dirs and
    # sort beside the baseline in TensorBoard at the parent-scenario level.
    run_tag = "curriculum" if curriculum_cfg else None
    variant_label = "curriculum" if curriculum_cfg else "baseline"

    print("\n" + "=" * 70)
    print(f"[DQN] {scenario} ({variant_label})  |  total_timesteps={total_ts:,}  |  seed={seed}")
    print("=" * 70)

    run_dir = new_run_dir(scenario, "dqn", seed=seed, tag=run_tag)
    # Persist the full, merged view of the YAML into config.json: the
    # hyperparams block also stores ``total_timesteps`` / ``n_envs`` so a
    # future reader doesn't have to join two sections to reconstruct the
    # training budget.
    write_config(
        run_dir,
        env=scenario,
        algo="dqn",
        seed=seed,
        hyperparams={**hp, "total_timesteps": total_ts, "n_envs": n_envs},
        env_settings=env_cfg,
        policy_kwargs=policy_cfg,
        gpu_setup=GPU_INFO,
        training_schedule={
            "checkpoint_freq": int(training_cfg.get("checkpoint_freq", 50_000)),
            "eval_freq": int(eval_cfg.get("eval_freq", 10_000)),
            "eval_episodes": int(eval_cfg.get("n_episodes", 10)),
        },
        curriculum=curriculum_cfg,
        variant=variant_label,
    )

    def _show(rd, scenario=scenario):
        # _record_video writes one mp4 per playthrough; list them all and
        # inline-display the first to keep notebook output manageable.
        vids = sorted((rd / "media").glob(f"dqn_{scenario}_ep*.mp4"))
        if not vids:
            vids = sorted((rd / "media").glob(f"dqn_{scenario}_ep*.gif"))
        if vids:
            for v in vids:
                print(f"[video] {v}")
            try:
                display(Video(str(vids[0]), embed=True))
            except Exception as exc:
                print(f"  (inline display skipped: {exc})")

    result = train_sb3(
        algo="dqn",
        scenario=scenario,
        run_dir=run_dir,
        hyperparams=hp,
        seed=seed,
        total_timesteps=total_ts,
        n_envs=n_envs,
        eval_freq=int(eval_cfg.get("eval_freq", 10_000)),
        eval_episodes=int(eval_cfg.get("n_episodes", 10)),
        checkpoint_freq=int(training_cfg.get("checkpoint_freq", 50_000)),
        record_video=True,
        device=DEVICE,
        on_complete=_show,
        resize_shape=tuple(env_cfg.get("resize_shape", (84, 84))),
        frame_skip=int(env_cfg.get("frame_skip", 4)),
        num_stack=int(env_cfg.get("num_stack", 4)),
        doom_skill=env_cfg.get("doom_skill"),
        num_bots=int(env_cfg.get("num_bots", 0)),
        policy_kwargs=policy_kwargs_from_config(policy_cfg),
        curriculum=curriculum_cfg,
    )
    results[scenario] = result
    print(
        f"[done] {scenario}: wall={result['wall_time_seconds']:.1f}s | "
        f"fps={result['fps']:.0f} | eval_mean={result['mean_eval_reward']}"
    )

print("\nAll DQN scenarios complete.")
for s, r in results.items():
    print(f"  - {s}: {r['run_dir']}")

## 4. Cross-scenario eval summary

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plt.figure(figsize=(10, 5))
for scenario, result in results.items():
    npz_path = Path(result["run_dir"]) / "metrics" / "training.npz"
    if not npz_path.exists():
        continue
    data = np.load(npz_path)
    eval_log = data["eval_rewards"]
    if eval_log.ndim == 2 and eval_log.shape[0] > 0:
        steps = eval_log[:, 0]
        means = eval_log[:, 1]
        stds = eval_log[:, 2]
        plt.plot(steps, means, marker="o", label=scenario)
        plt.fill_between(steps, means - stds, means + stds, alpha=0.2)
plt.xlabel("Environment Steps")
plt.ylabel("Eval Reward")
plt.title("DQN — Cross-scenario eval")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Disconnect runtime

In [ ]:
# Disconnect the Colab runtime at the end of the notebook to save compute.
# No-op when running locally.
try:
    from google.colab import runtime as _colab_runtime
except ImportError:
    pass
else:
    import time
    print("Notebook finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    _colab_runtime.unassign()